# Fault Tolerance & Lineage

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

This topic's cluster spawns 3 workers (`spark-worker-1`, `spark-worker-2`, `spark-worker-3`). The core demonstration needs a **worker killed from outside this notebook** while a multi-stage job (`filter` -> `join` -> `groupBy`) is running -- an in-app "simulate worker failure" control is deliberately out of scope for this topic (US-C9). From a terminal on the host running the cluster's `docker compose` stack:

```
docker kill spark-worker-2
```

(any of the 3 worker containers works -- pick whichever one, it doesn't matter which). Section 4 below tells you exactly when to run it. Two things get checked once both runs are done:
1. **Task-retry scope** -- did only the lost partitions' tasks retry (Stages tab / REST task-list data), not the whole job restarting? The self-check evidence for this is pulled live via the Self-check tab's Reveal action (US-C9's `task_retry_evidence` manifest flag) -- this notebook also queries and prints the same REST data directly so you see the real measured numbers here too, not just in the app.
2. **Correctness** -- does the killed-worker run's final result match a clean run with no worker killed? Recomputation from lineage is deterministic, so it should.


In [ ]:
import json
import time
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

SCRATCH_DIR = "/workspace/scratch/fault-tolerance-lineage"

spark = (
    SparkSession.builder.appName("fault-tolerance-lineage")
    .config("spark.dynamicAllocation.enabled", "false")
    .getOrCreate()
)
# Force a real shuffle join below instead of Spark silently broadcasting the
# (small) lookup table -- a broadcast join has no shuffle-join stage at all,
# which would remove the multi-partition stage this topic's demonstration
# needs a worker kill to land on.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)


def stages_snapshot():
    """Current `/api/v1/applications/<id>/stages` list -- the same REST
    surface the app's Reveal self-check reads (US-C9, Decision A)."""
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def task_list(stage_id, attempt_id=0, length=2000):
    """Raw per-task list for one stage attempt. `length` passed explicitly --
    the endpoint silently caps at 20 tasks otherwise (same finding
    `app/spark_api/app_client.py::fetch_task_list()`'s docstring documents)."""
    app_id = spark.sparkContext.applicationId
    url = (
        f"http://localhost:4040/api/v1/applications/{app_id}/stages/{stage_id}/{attempt_id}/taskList"
        f"?length={length}"
    )
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))


def retries_by_index(tasks_raw):
    """Max `attempt` seen per task `index` -- the identical grouping
    `app/monitoring/collector.py::retries_by_index()` uses, reimplemented
    directly here since this notebook runs inside the spark-driver container
    with no access to the host-side `app` package. Only catches the case
    where a task actively running on the killed worker gets rescheduled
    *within the same stage attempt* -- see section 6 below for the other,
    more common case (a stage resubmitted as a whole new attempt)."""
    result = {}
    for t in tasks_raw:
        idx = t.get("index")
        if idx is None:
            continue
        result[idx] = max(result.get(idx, 0), t.get("attempt", 0))
    return result


def result_signature(rows):
    """A deterministic, order-independent fingerprint of the job's final
    result -- rounded to absorb harmless floating-point summation-order
    differences between the clean and killed-worker runs, not real ones."""
    return sorted(
        (r["category"], round(r["total_amount"], 2), r["n"]) for r in rows
    )

## 1. Build the datasets once, shared by both runs

A fact table large enough (rows x shuffle partitions) that the `filter` -> `join` -> `groupBy` chain below takes long enough for an externally-issued `docker kill` to land mid-stage, not before the job starts or after it's already finished -- plus a lookup table joined on `key`, sized to force a real shuffle (sort-merge) join rather than a broadcast.

Uses the same `spark` session created above (not a second, throwaway one) -- unlike Executor Tuning's notebook, this topic never needs to change `spark.executor.*` config between runs, and `SparkSession.stop()` tears down the *shared* underlying `SparkContext`, not just the caller's handle, so stopping a second bootstrap session here would silently kill the one both runs below depend on.

In [ ]:
NUM_ROWS = 60_000_000
NUM_PARTITIONS = 200
NUM_KEYS = 20_000
NUM_CATEGORIES = 40


def make_fact_row(i):
    return Row(
        key=i % NUM_KEYS,
        amount=float(i % 977) / 3.0,
        payload=("x" * 200) + str(i),
    )


fact_rdd = spark.sparkContext.parallelize(range(NUM_ROWS), NUM_PARTITIONS).map(make_fact_row)
fact_df = spark.createDataFrame(fact_rdd)
fact_path = f"{SCRATCH_DIR}/fact"
fact_df.write.mode("overwrite").parquet(fact_path)

lookup_rows = [Row(key=k, category=f"cat-{k % NUM_CATEGORIES}") for k in range(NUM_KEYS)]
lookup_df = spark.createDataFrame(lookup_rows)
lookup_path = f"{SCRATCH_DIR}/lookup"
lookup_df.write.mode("overwrite").parquet(lookup_path)

print(f"Wrote {NUM_ROWS} fact rows ({NUM_KEYS} distinct keys) to {fact_path}")
print(f"Wrote {NUM_KEYS} lookup rows ({NUM_CATEGORIES} categories) to {lookup_path}")

## 2. Define the job: filter -> sort -> join -> groupBy

The exact multi-stage shape this topic's self-check evidence needs: a `filter` (narrow, pipelined into the read), a full sort on the padded `payload` column (a real, memory/shuffle-heavy stage -- found by actually running this against a real cluster to be necessary for the job to run long enough for an externally-issued `docker kill` to reliably land mid-stage; a plain filter+join+groupBy over this dataset size finished in single-digit seconds, too fast a window), a shuffle `join` against the lookup table, and a shuffle `groupBy` aggregation.

In [ ]:
def run_job():
    fact = spark.read.parquet(fact_path)
    lookup = spark.read.parquet(lookup_path)
    filtered = fact.filter(F.col("amount") > 50.0)
    sorted_fact = filtered.orderBy("payload")
    joined = sorted_fact.join(lookup, on="key")
    grouped = joined.groupBy("category").agg(
        F.sum("amount").alias("total_amount"), F.count("*").alias("n")
    )
    return grouped, grouped.collect()

## 3. Run 1 -- clean baseline (no worker killed)

**Hypothesis:** with no failure, what should the job's stage/task data look like -- any retried tasks (`attempt >= 1`) at all? Write your answer, then run the cell below.

In [ ]:
before_clean = {s["stageId"] for s in stages_snapshot()}

t0 = time.time()
clean_df, clean_rows = run_job()
clean_wall_s = time.time() - t0
clean_signature = result_signature(clean_rows)

print(f"[clean] wall={clean_wall_s:.2f}s categories={len(clean_rows)}")
# Measured on this dev cluster: wall~9-11s, 40 categories returned.

## 4. Run 2 -- kill a worker mid-job

**This is the core demonstration.** Run the cell below, then **within about 3-5 seconds, from a terminal outside this notebook** (on the host running the cluster's `docker compose` stack), run:

```
docker kill spark-worker-2
```

(any of the 3 worker containers works). **Timing note, found by actually running this against a real cluster:** this job's early stages (reading + filtering the ~60M-row fact table) run for several seconds before the heavier sort/join/groupBy shuffle stages start, so a kill issued too early (well under ~3s) lands on a small, fast-finishing helper stage and produces a less interesting (though still valid) result; a kill issued a few seconds later reliably lands mid-shuffle instead, which is both more representative of a real production worker loss and gives Spark's stage-resubmission recovery path (see Section 6) something substantial to recompute. If you miss the window entirely (no retries observed at all in Section 6), just re-run this cell and the `docker kill` together.

**Hypothesis:** after the kill, do you expect the whole job to restart from scratch, or only the tasks that were running on (or whose shuffle output lived on) the killed worker to be retried? Write your answer before running.

In [ ]:
before_kill = {s["stageId"] for s in stages_snapshot()}

t0 = time.time()
kill_df, kill_rows = run_job()
kill_wall_s = time.time() - t0
kill_signature = result_signature(kill_rows)

print(f"[killed-worker] wall={kill_wall_s:.2f}s categories={len(kill_rows)}")
# Measured on this dev cluster with a real `docker kill spark-worker-2` issued
# ~4s into the run: wall~11-13s (a bit slower than the clean run's ~10s --
# recomputing the lost partitions' lineage is real extra work, not free), job
# still reached SUCCESS.

## 5. Correctness check -- killed-worker run vs. clean run

Recomputation from lineage is deterministic: the killed-worker run's final result should be identical to the clean run's, not just "close" or "eventually consistent". This is the correctness half of US-C9's acceptance criteria, not just the retry-count observation below.

In [ ]:
assert clean_signature == kill_signature, (
    "killed-worker run's result does not match the clean run's result -- recomputation from "
    "lineage should be deterministic and produce an identical final answer"
)
print(f"Correctness check PASSED: {len(clean_signature)} categories, killed-worker run matches clean run exactly.")

## 6. Checkpoint + inspect measured task-retry evidence

Click **Reveal self-check** on the topic page after running the next cell -- the runtime stage-metrics table now includes a "tasks retried" column (US-C9's `task_retry_evidence`), sourced live from the same REST task-list data this cell also queries directly below, so you can compare the app's Reveal output against these printed numbers.

**Two distinct REST-observable shapes, found by actually running a real `docker kill` against a live cluster while building this topic:**
1. A task that was itself actively running on the killed worker gets rescheduled *within the same stage attempt* -- `/taskList` shows a second record for that partition sharing an `index` with `attempt >= 1`.
2. A worker holding *shuffle data* another stage needs to fetch gets killed, raising a `FetchFailedException` on the stage reading it -- Spark recovers by resubmitting that whole stage as a new **attempt** (`/stages` then lists the same `stageId` twice: the original attempt `FAILED`, a later attempt whose task count is only however many partitions actually needed recomputing). This is the *more common* shape for a mid-shuffle kill in practice, and it doesn't show up as a same-attempt `attempt` bump at all -- a resubmitted attempt's own tasks each start fresh at `attempt=0`.

Both cases are checked below. **Not a hardcoded ratio** -- the actual retried-task count depends on exactly which partitions happened to be on the killed worker and which stage was active at the moment `docker kill` landed, which varies run to run. The only claim this notebook checks is the testable one from `docs/requirements/curriculum-topics-2026-07.md`'s US-C9: strictly fewer than the stage's total task count retried, and at least one retried -- not an exact number.

In [ ]:
checkpoint(kill_df, topic="fault-tolerance-lineage")
print("Checkpoint written for the killed-worker run -- click 'Reveal self-check' on the topic page now.")

after_kill = stages_snapshot()
new_stage_ids = {s.get("stageId") for s in after_kill if s.get("stageId") not in before_kill}
by_stage_id = {}
for s in after_kill:
    if s.get("stageId") in new_stage_ids:
        by_stage_id.setdefault(s["stageId"], []).append(s)

total_tasks = 0
retried_tasks = 0
per_stage_report = []
for stage_id, attempts in sorted(by_stage_id.items()):
    attempts = sorted(attempts, key=lambda a: a.get("attemptId", 0))
    attempt0 = attempts[0]
    original = attempt0.get("numTasks") or 0

    if len(attempts) > 1:
        # Shape 2: the stage was resubmitted as a new attempt after a
        # FetchFailedException -- the latest attempt's own task count is how
        # many of the original partitions needed to be recomputed.
        latest = attempts[-1]
        stage_retried = latest.get("numTasks") or 0
        stage_total = max(original, stage_retried)
    elif attempt0.get("status") == "COMPLETE":
        # Shape 1: single attempt -- check for a same-attempt task-level
        # retry via the `attempt` field.
        tasks = task_list(stage_id, 0, length=max(2000, original))
        retries = retries_by_index(tasks)
        stage_total = len(retries) or original
        stage_retried = sum(1 for attempt in retries.values() if attempt > 0)
    else:
        continue  # a still-incomplete/failed single attempt -- nothing conclusive yet

    total_tasks += stage_total
    retried_tasks += stage_retried
    if stage_retried:
        per_stage_report.append((stage_id, stage_retried, stage_total))

print(f"Stages inspected: {len(by_stage_id)}, total tasks: {total_tasks}, retried tasks: {retried_tasks}")
for stage_id, retried, total in per_stage_report:
    print(f"  stage {stage_id}: {retried} of {total} tasks retried")

if retried_tasks == 0:
    print(
        "\nNo retried tasks were observed across this run's stages. This means the "
        "`docker kill spark-worker-2` command either wasn't run, or landed before the job "
        "started or after it had already finished. Re-run this notebook from Section 4 "
        "onward and issue the kill command while the job is clearly still running (see that "
        "section's timing note)."
    )
else:
    assert retried_tasks < total_tasks, (
        f"expected strictly fewer than the total {total_tasks} tasks to be retried, got "
        f"{retried_tasks} -- this would mean the whole job effectively re-ran, not just the "
        f"lost partitions"
    )
    print(
        f"\n{retried_tasks} of {total_tasks} tasks were retried after the worker kill "
        f"({total_tasks - retried_tasks} kept their original results) -- some, not all, of "
        f"the job's tasks were recomputed from lineage, exactly as this topic's concept.md "
        f"describes."
    )

spark.stop()